# 04 — RAG Pipeline: Complete Theory & Implementation

## Part 1: Theory

### 1.1 Why RAG Exists

LLMs have two fundamental problems:
1. **Hallucination**: Confidently generate false information
2. **Knowledge cutoff**: Can't access information after training date

RAG solves both by **retrieving relevant context** before generating.

### 1.2 RAG vs Fine-Tuning vs Long Context

| Approach | When to Use | Cost | Freshness | Accuracy |
|----------|------------|------|-----------|----------|
| **RAG** | Dynamic knowledge, many docs | Low (no training) | Real-time | High (grounded) |
| **Fine-tuning** | Behavioral changes, style | High (GPU hours) | Static | Medium |
| **Long context** | Few docs, simple lookup | Medium (tokens) | Per-request | Variable |

### 1.3 RAG Architecture Evolution

```
Naive RAG:     Query → Retrieve → Generate
Advanced RAG:  Query → Rewrite → Retrieve → Rerank → Generate
Modular RAG:   Query → Route → [Retrieve|Search|SQL] → Filter → Rerank → Generate → Verify
```

### 1.4 Retrieval Strategies

| Strategy | How | Pros | Cons |
|----------|-----|------|------|
| Dense | Embed query, cosine search | Semantic understanding | Misses keywords |
| Sparse (BM25) | TF-IDF keyword match | Exact term matching | No semantics |
| Hybrid | Dense + Sparse + RRF fusion | Best of both | More complex |
| Multi-vector | Summary + full chunk embeddings | Better matching | 2x storage |
| Parent-child | Small chunks for search, return parent | Precise + contextual | Complex indexing |

### 1.5 Re-ranking

Initial retrieval (bi-encoder) is fast but imprecise. **Cross-encoders** re-score top-k results:
```
Bi-encoder:    score = cosine(embed(query), embed(doc))     ← independent encoding
Cross-encoder: score = model([query, doc])                   ← joint encoding (slower, better)
```

### 1.6 RAG Failure Modes

| Failure | Symptom | Fix |
|---------|---------|-----|
| Retrieval miss | Correct answer not in context | Better chunking, hybrid search |
| Lost in the middle | Model ignores middle context | Rerank, put key info first/last |
| Context poisoning | Irrelevant docs confuse model | Relevance filtering, threshold |
| Hallucination despite context | Model ignores retrieved info | Stronger system prompt, lower temp |

### 1.7 RAG Evaluation (RAGAS Framework)

| Metric | Measures | How |
|--------|----------|-----|
| Faithfulness | Is answer grounded in context? | Check claims against retrieved docs |
| Answer Relevance | Does answer address the question? | Semantic sim(answer, question) |
| Context Recall | Did we retrieve the right docs? | Ground truth coverage |
| Context Precision | Are retrieved docs relevant? | Relevant docs / total retrieved |

---

### Architecture: Full RAG Pipeline
```
INDEXING (offline):                    QUERYING (online):
┌──────┐   ┌───────┐   ┌───────┐     ┌───────┐   ┌──────────┐   ┌─────────┐   ┌──────────┐
│ Docs │──▶│ Chunk │──▶│ Embed │──▶   │ Query │──▶│ Retrieve │──▶│ Rerank  │──▶│ Generate │
└──────┘   └───────┘   └───┬───┘     └───────┘   └────┬─────┘   └─────────┘   └──────────┘
                            │                          │
                            ▼                          │
                       ┌─────────┐                     │
                       │ Vector  │◀────────────────────┘
                       │   DB    │
                       └─────────┘
```

## Part 2: Implementation

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

OPENROUTER_BASE = "https://openrouter.ai/api/v1"
OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")

llm = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct:free",
    openai_api_key=OPENROUTER_KEY,
    openai_api_base=OPENROUTER_BASE,
    temperature=0,
)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("Setup complete")

### Document Loading & Chunking

In [ ]:
raw_documents = [
    """RAG Architecture: Retrieval-Augmented Generation enhances LLM responses by retrieving relevant context from external knowledge bases. The pipeline: 1) Document ingestion and chunking, 2) Embedding generation, 3) Vector storage, 4) Query-time retrieval, 5) Context-augmented generation. RAG reduces hallucinations by grounding responses in factual data.""",
    """Chunking Strategies: Effective chunking is critical for RAG quality. Fixed-size (simple but splits sentences), Recursive (respects structure), Semantic (groups by meaning), Document-aware (respects headers). Optimal: 500-1000 tokens with 50-100 overlap.""",
    """Embedding Models: text-embedding-3-small (1536d, good quality/cost), all-MiniLM-L6-v2 (384d, free, fast), BGE-large (1024d, best open-source), Cohere embed-v3 (1024d, multilingual).""",
    """Retrieval Optimization: 1) Hybrid search (dense + sparse), 2) Re-ranking with cross-encoders, 3) Query expansion, 4) Metadata filtering, 5) Multi-vector retrieval. Measure with recall@k and MRR.""",
    """Production RAG: Needs caching for repeated queries, rate limiting, fallback strategies, document freshness tracking, access control, and observability for debugging.""",
    """Advanced RAG Patterns: HyDE generates hypothetical answers to improve retrieval. Multi-query generates multiple search queries from one question. Self-RAG decides when to retrieve. CRAG checks retrieval quality and falls back to web search.""",
    """RAG Evaluation: Use RAGAS framework - Faithfulness (grounded in context?), Answer Relevance (addresses question?), Context Recall (right docs retrieved?), Context Precision (retrieved docs relevant?). Aim for >0.8 on all.""",
]

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.create_documents(raw_documents)
print(f"Split {len(raw_documents)} docs → {len(chunks)} chunks")

### Build Vector Store & Basic RAG Chain

In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_prompt = ChatPromptTemplate.from_template(
    """Answer based only on the context below. If unsure, say so.

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test
questions = ["What is RAG?", "What chunking size should I use?", "How does HyDE work?"]
for q in questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}\nA: {answer}\n")

### Conversational RAG with Memory

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the follow-up question as a standalone question using the chat history for context."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

def conversational_rag(question, chat_history):
    if chat_history:
        standalone = (contextualize_prompt | llm | StrOutputParser()).invoke(
            {"chat_history": chat_history, "input": question}
        )
    else:
        standalone = question
    docs = retriever.invoke(standalone)
    context = format_docs(docs)
    answer = (rag_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    return answer

# Multi-turn demo
history = []
for q in ["What is RAG?", "How do I evaluate it?", "What scores should I aim for?"]:
    a = conversational_rag(q, history)
    print(f"Q: {q}\nA: {a}\n")
    history.extend([HumanMessage(content=q), AIMessage(content=a)])

### RAG Evaluation

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

eval_embedder = SentenceTransformer("all-MiniLM-L6-v2")

eval_set = [
    {"q": "What is RAG?", "expected": "RAG combines retrieval with generation to ground LLM responses in external knowledge."},
    {"q": "What chunk size should I use?", "expected": "500-1000 tokens with 50-100 token overlap."},
    {"q": "How to evaluate RAG?", "expected": "Use RAGAS: faithfulness, relevance, context recall, context precision."},
    {"q": "What is HyDE?", "expected": "HyDE generates hypothetical answers to improve retrieval quality."},
]

results = []
for item in eval_set:
    answer = rag_chain.invoke(item["q"])
    embs = eval_embedder.encode([answer, item["expected"]])
    sim = cos_sim([embs[0]], [embs[1]])[0][0]
    results.append({"question": item["q"], "similarity": round(sim, 4), "answer_len": len(answer)})

import pandas as pd
eval_df = pd.DataFrame(results)
print(f"Average accuracy: {eval_df['similarity'].mean():.4f}")
eval_df

## Part 3: Key Takeaways

1. **RAG = Retrieve + Generate** — grounds LLMs in facts, eliminates hallucination
2. **Chunking quality** directly determines retrieval quality
3. **Hybrid search** (dense + sparse) outperforms either alone
4. **Conversational RAG** needs question reformulation for follow-ups
5. **Evaluate systematically** with RAGAS-style metrics, not vibes
6. **Production RAG** needs caching, fallbacks, and observability

### Next → 05_mlops.ipynb